# Reddit data, three realistic routes

The Pushshift live API closed in 2023 and Reddit's API became paid, so "get Reddit data"
now means choosing among three routes. This starter demonstrates all three and runs
offline on built-in samples, so you can read the whole pipeline before touching a key.

**Licensing tier (see `kits/licensing-one-pager.md`):** analyze and publish findings;
do not redistribute post text as a dataset. Deleted posts stay deleted in your work.

> **Don't lose your work.** Opened from GitHub, this notebook is read-only: **File → Save a copy in Drive** before editing, and save durable outputs to your Drive project folder; Colab's own disk is wiped when the runtime ends. Course notebooks get updates during the term; to pick them up, open the notebook fresh from GitHub (or `git pull` if you cloned the repo). Updates never touch your saved copy.


In [ ]:
import os, io
import pandas as pd

SAMPLE = pd.DataFrame([
    {"subreddit": "AmItheAsshole", "title": "AITA for leaving the wedding early?", "score": 4120, "created": "2021-06-12", "verdict": "NTA"},
    {"subreddit": "AmItheAsshole", "title": "AITA for correcting my boss in a meeting?", "score": 2874, "created": "2021-07-03", "verdict": "ESH"},
    {"subreddit": "AmItheAsshole", "title": "AITA for not sharing my lottery win?", "score": 9911, "created": "2021-08-19", "verdict": "NTA"},
    {"subreddit": "AmItheAsshole", "title": "AITA for renaming the office plant?", "score": 133, "created": "2021-09-02", "verdict": "YTA"},
])
print("offline sample:", len(SAMPLE), "posts")

## Route 1: historical archives (2005–2022), the default for most projects

The Pushshift monthly dumps survive as Hugging Face mirrors and academic torrents
(Baumgartner et al., ICWSM 2020, is the corpus paper). One `load_dataset` call gives you
years of a subreddit with scores and timestamps; no key, no rate limit, reproducible.

In [ ]:
def load_historical(dataset_id, split="train", n=1000):
    """Load a Reddit dump mirror from Hugging Face; on any failure, the sample."""
    try:
        from datasets import load_dataset
        ds = load_dataset(dataset_id, split=f"{split}[:{n}]")
        return ds.to_pandas()
    except Exception as e:
        print("no network or dataset unavailable, using the sample:", e)
        return SAMPLE

# Example mirrors to search on huggingface.co/datasets: "pushshift", "reddit AITA",
# "webis/tldr-17" (3.8M posts, clean license). Archives are large; start the real pull
# only once you have chosen one:
# df = load_historical("webis/tldr-17")
df = SAMPLE
df.head()

## Route 2: the official API via PRAW, for small current samples

Reddit's free OAuth tier still permits small-scale, non-commercial reads (about 100
queries per minute, and listings cap near 1,000 posts). Register a script app at
reddit.com/prefs/apps, then store the credentials as environment variables or Colab
Secrets, never in the notebook.

In [ ]:
REDDIT_ID = os.environ.get("REDDIT_CLIENT_ID")
REDDIT_SECRET = os.environ.get("REDDIT_CLIENT_SECRET")
LIVE = bool(REDDIT_ID and REDDIT_SECRET)

def fetch_recent(subreddit, limit=100):
    if not LIVE:
        print("no Reddit credentials set, returning the offline sample")
        return SAMPLE
    import praw  # needs: pip install praw
    reddit = praw.Reddit(client_id=REDDIT_ID, client_secret=REDDIT_SECRET,
                         user_agent="culture-as-data course project (small-scale, non-commercial)")
    rows = [{"subreddit": subreddit, "title": p.title, "score": p.score,
             "created": pd.to_datetime(p.created_utc, unit="s"), "text": p.selftext}
            for p in reddit.subreddit(subreddit).top(time_filter="year", limit=limit)]
    return pd.DataFrame(rows)

recent = fetch_recent("AmItheAsshole")
print(len(recent), "posts")

## Route 3: the Reddit-fiction loader (r/nosleep, r/HFY)

Born-digital fiction is the cleanest path for horror and science-fiction projects:
contemporary, publicly posted, and reachable by either route above. This wrapper pulls
story-length posts only.

In [ ]:
def fetch_fiction(subreddit="nosleep", limit=100, min_chars=2000):
    """Story-length posts from a fiction subreddit; offline, a built-in excerpt."""
    if not LIVE:
        return pd.DataFrame([{
            "subreddit": subreddit,
            "title": "The stairwell only goes down",
            "text": ("Nobody else in the building will talk about the stairwell. " * 60),
            "score": 5321}])
    df = fetch_recent(subreddit, limit=limit)
    stories = df[df["text"].str.len() >= min_chars].copy()
    return stories

stories = fetch_fiction("nosleep")
print(len(stories), "stories;", int(stories["text"].str.len().mean()), "chars on average")

## Save to your project folder

Whichever route: save once, load from Drive after. Your analysis should never depend
on the network twice.

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive"); PROJECT_DIR = "/content/drive/MyDrive/culture-as-data"
except Exception:
    PROJECT_DIR = os.path.abspath("./culture-as-data-project")
os.makedirs(PROJECT_DIR, exist_ok=True)
df.to_csv(os.path.join(PROJECT_DIR, "reddit_corpus.csv"), index=False)
print("saved ->", os.path.join(PROJECT_DIR, "reddit_corpus.csv"))